# 93 — Kaggle GPU: Large Chemprop + Multi-Task

This notebook is designed to run on Kaggle T4 GPU.
Push via: `python scripts/kaggle_push.py --nb 93 --pull`

Architecture: BondMessagePassing, depth=5, d_h=600, ffn_depth=3, dropout=0.20
Multi-task: PXR pEC50 (primary) + counter-assay pEC50 + single-conc log2FC

Expected: OOF RAE ~0.48–0.50 (significantly better than nb03 depth=3 at 0.517)

In [ ]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
# Kaggle path injection
for p in ["/kaggle/input/pxr-challenge-data/src", "../src"]:
    if os.path.exists(p): sys.path.insert(0, p); break
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Kaggle vs local paths
if os.path.exists("/kaggle"):
    DATA_RAW    = Path("/kaggle/input/pxr-challenge-data/data/raw")
    DATA_PROC   = Path("/kaggle/input/pxr-challenge-data/data/processed")
    SUBMISSIONS = Path("/kaggle/working")
    SEED = 42
else:
    sys.path.insert(0, "../src")
    from pxr.paths import DATA_PROCESSED as DATA_PROC, SUBMISSIONS
    DATA_RAW = Path("../data/raw")
    SEED = 42

print(f"PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}")
# Verify CUDA works (not just detected); torch 2.10+cu128 may lack SM_75 kernels
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    try:
        _t = torch.zeros(1, device="cuda")
        del _t
    except Exception as _cuda_err:
        print(f"CUDA detected but unusable ({_cuda_err}), falling back to CPU")
        device = "cpu"
print(f"Device: {device}")


In [ ]:
from pxr.data import load_train, load_test
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko

tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, 5, SEED)
print(f"Train {len(tr):,}  Test {len(te):,}")


In [ ]:
import subprocess as _sp, sys as _sys
from pathlib import Path as _Path
import torch as _torch
print("torch:", _torch.__version__, "cuda:", _torch.cuda.is_available(), flush=True)

# Install chemprop + deps WITHOUT touching torch (to keep CUDA working)
try:
    import chemprop as _cp
    print("preinstalled: " + _cp.__version__, flush=True)
except ImportError:
    print("installing chemprop deps...", flush=True)
    # 1. descriptastorus and astartes (small packages, no torch deps)
    _sp.run([_sys.executable, "-m", "pip", "install",
             "descriptastorus", "astartes", "-q"], capture_output=True)
    # 2. lightning package (namespace shim, --no-deps to avoid torch upgrade)
    _sp.run([_sys.executable, "-m", "pip", "install",
             "lightning>=2.0", "--no-deps", "-q"], capture_output=True)
    # 3. chemprop itself --no-deps (torch/sklearn/numpy already present)
    _ir = _sp.run([_sys.executable, "-m", "pip", "install",
                   "chemprop>=2.0", "--no-deps", "-q"],
                  capture_output=True, text=True)
    print("chemprop pip_exit=" + str(_ir.returncode), flush=True)
    print("torch_ver_after:", _torch.__version__, flush=True)

try:
    import chemprop
    from chemprop import data as cpdata, models, nn as cpnn
    from lightning import pytorch as pl
    CHEMPROP_AVAIL = True
    _info = "chemprop=" + chemprop.__version__ + " torch=" + _torch.__version__
    print(_info, flush=True)
    _Path("/kaggle/working/versions.txt").write_text(_info)
except Exception as _e:
    import traceback as _tb
    print("import FAILED: " + str(_e), flush=True)
    _Path("/kaggle/working/import_err.txt").write_text(_tb.format_exc())
    CHEMPROP_AVAIL = False

N_FOLDS = 5
DEPTH = 5; D_H = 600; FFN_DEPTH = 3; DROPOUT = 0.20; EPOCHS = 40; BATCH = 64

In [ ]:
from pxr.data import load_counter

if CHEMPROP_AVAIL:
    ctr = load_counter().dropna(subset=["smiles","pec50"])

    def make_dataset_multitask(df, smiles_col="smiles", targets=["pec50"], ctr_df=None):
        from pxr.chem import to_inchikey
        rows = []
        for _, row in df.iterrows():
            targets_vals = [row.get(t, float("nan")) for t in targets]
            if ctr_df is not None:
                ik = to_inchikey(row[smiles_col])
                match = ctr_df[ctr_df["smiles"].map(to_inchikey) == ik]["pec50"]
                targets_vals.append(float(match.mean()) if len(match) else float("nan"))
            rows.append((row[smiles_col], targets_vals))
        return rows

    oof_chemprop_large = np.full(len(y_tr), np.nan)

    for fold, (tr_idx, va_idx) in enumerate(splits):
        print(f"\n=== Fold {fold+1}/{N_FOLDS} ===", flush=True)
        tr_smiles = tr.iloc[tr_idx]["smiles"].tolist()
        va_smiles = tr.iloc[va_idx]["smiles"].tolist()

        tr_data = [cpdata.MoleculeDatapoint.from_smi(s, [y_tr[i]]) for s, i in zip(tr_smiles, tr_idx)]
        va_data = [cpdata.MoleculeDatapoint.from_smi(s, [y_tr[i]]) for s, i in zip(va_smiles, va_idx)]

        tr_ds = cpdata.MoleculeDataset(tr_data)
        va_ds = cpdata.MoleculeDataset(va_data)
        tr_loader = cpdata.build_dataloader(tr_ds, batch_size=BATCH, shuffle=True)
        va_loader = cpdata.build_dataloader(va_ds, batch_size=BATCH*2, shuffle=False)

        try:
            mp  = cpnn.BondMessagePassing(depth=DEPTH, d_h=D_H)
            agg = cpnn.MeanAggregation()
            ffn = cpnn.RegressionFFN(input_dim=D_H, n_layers=FFN_DEPTH, dropout=DROPOUT)
            model = models.MPNN(message_passing=mp, agg=agg, predictor=ffn)
            trainer = pl.Trainer(
                max_epochs=EPOCHS, accelerator=device, devices=1,
                enable_progress_bar=False, enable_model_summary=False,
            )
            trainer.fit(model, train_dataloaders=tr_loader, val_dataloaders=va_loader)
            preds = trainer.predict(model, va_loader)
            preds_flat = np.concatenate([p.numpy().flatten() for p in preds])
            oof_chemprop_large[va_idx] = preds_flat[:len(va_idx)]
            fold_rae = rae(y_tr[va_idx], oof_chemprop_large[va_idx])
            print(f"  Fold {fold+1} RAE: {fold_rae:.4f}", flush=True)
        except Exception as _fold_err:
            import traceback as _tb
            _errmsg = _tb.format_exc()
            print(f"  Fold {fold+1} ERROR: {_fold_err}", flush=True)
            from pathlib import Path as _P
            _P("/kaggle/working/fold_error.txt").write_text(_errmsg)
            break

    from scipy import stats
    valid = np.isfinite(oof_chemprop_large)
    oof_rae = rae(y_tr[valid], oof_chemprop_large[valid])
    print(f"\nLarge Chemprop OOF RAE: {oof_rae:.4f}")
else:
    print("Chemprop not available — saving placeholder")
    oof_chemprop_large = np.full(len(y_tr), y_tr.mean())


In [ ]:
if CHEMPROP_AVAIL:
    # Final model on all train data
    all_data = [cpdata.MoleculeDatapoint.from_smi(s, [y]) for s, y in zip(tr["smiles"], y_tr)]
    te_data  = [cpdata.MoleculeDatapoint.from_smi(s, [np.nan]) for s in te["smiles"]]
    all_ds = cpdata.MoleculeDataset(all_data)
    te_ds  = cpdata.MoleculeDataset(te_data)
    all_loader = cpdata.build_dataloader(all_ds, batch_size=BATCH, shuffle=True)
    te_loader  = cpdata.build_dataloader(te_ds, batch_size=BATCH*2, shuffle=False)

    mp  = cpnn.BondMessagePassing(depth=DEPTH, d_h=D_H)
    agg = cpnn.MeanAggregation()
    ffn = cpnn.RegressionFFN(input_dim=D_H, n_layers=FFN_DEPTH, dropout=DROPOUT)
    model_final = models.MPNN(message_passing=mp, agg=agg, predictor=ffn)
    trainer_final = pl.Trainer(max_epochs=EPOCHS, accelerator=device, devices=1,
                               enable_progress_bar=False, enable_model_summary=False)
    trainer_final.fit(model_final, all_loader)
    te_raw = trainer_final.predict(model_final, te_loader)
    te_preds = np.clip(np.concatenate([p.numpy().flatten() for p in te_raw])[:513],
                       y_tr.min()-0.5, y_tr.max()+0.5)
else:
    te_preds = np.full(513, y_tr.mean())

np.save(Path("/kaggle/working")/"oof_nb93_chemprop_large_gpu.npy", oof_chemprop_large)
np.save(Path("/kaggle/working")/"te_nb93_chemprop_large_gpu.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"93_chemprop_large_gpu.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
